# Task 4: Imbalance Handling

Comparing three approaches to handling the ~80/20 class imbalance in the target variable:
- `scale_pos_weight` — reweights the loss function, no data changes
- `is_unbalance=True` — LightGBM's automatic reweighting
- SMOTE — synthetic oversampling of the minority class

All models use identical settings otherwise (`random_state=42`, `n_estimators=100`, `deterministic=True`, `force_row_wise=True`) so differences are attributable only to the imbalance technique.

In [1]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import joblib
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, confusion_matrix, classification_report

pd.set_option('display.max_columns', None)

In [2]:
df = pd.read_csv('../data/processed/loan_data_engineered.csv')

In [3]:
drop_cols = ['loan_status', 'default', 'earliest_cr_line']
X = df.drop(columns=drop_cols)
y = df['default']

In [4]:
categorical_cols = X.select_dtypes(include='object').columns.tolist()
for col in categorical_cols:
    X[col] = X[col].astype('category')

C:\Users\ARYAN DASH\AppData\Local\Temp\ipykernel_13268\3967489894.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X.select_dtypes(include='object').columns.tolist()


In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [6]:
print(X_train.shape, X_test.shape)
print(y_train.value_counts())

(1076280, 47) (269070, 47)
default
0    861401
1    214879
Name: count, dtype: int64


## Approach 1: scale_pos_weight

scale_pos_weight = count(negative) / count(positive) ≈ 4.01

Penalizes misclassifying a default ~4x more heavily during training. No changes to the underlying data.

In [8]:
scale_pos_weight_value = y_train.value_counts()[0] / y_train.value_counts()[1]
print(f"scale_pos_weight: {scale_pos_weight_value:.4f}")

scale_pos_weight: 4.0088


In [9]:
model_spw = lgb.LGBMClassifier(
    random_state=42,
    n_estimators=100,
    deterministic=True,
    force_row_wise=True,
    scale_pos_weight=scale_pos_weight_value
)

In [10]:
model_spw.fit(X_train, y_train, categorical_feature=categorical_cols)

[LightGBM] [Info] Number of positive: 214879, number of negative: 861401
[LightGBM] [Info] Total Bins 5648
[LightGBM] [Info] Number of data points in the train set: 1076280, number of used features: 47
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.199650 -> initscore=-1.388485
[LightGBM] [Info] Start training from score -1.388485


,random_state,42
,deterministic,True
,force_row_wise,True
,scale_pos_weight,np.float64(4.008772378873692)
,boosting_type,'gbdt'
,num_leaves,31
,max_depth,-1
,learning_rate,0.1
,n_estimators,100
,subsample_for_bin,200000
,objective,None


In [11]:
y_pred_proba_spw = model_spw.predict_proba(X_test)[:, 1]
y_pred_spw = model_spw.predict(X_test)

In [13]:
auc_spw = roc_auc_score(y_test, y_pred_proba_spw)
print(f"AUC: {auc_spw:.4f}")
print("\n")
print(confusion_matrix(y_test, y_pred_spw))
print("\n")
print(classification_report(y_test, y_pred_spw))

AUC: 0.7224


[[139306  76044]
 [ 17573  36147]]


              precision    recall  f1-score   support

           0       0.89      0.65      0.75    215350
           1       0.32      0.67      0.44     53720

    accuracy                           0.65    269070
   macro avg       0.61      0.66      0.59    269070
weighted avg       0.78      0.65      0.69    269070



## Approach 2: is_unbalance=True

LightGBM's built-in automatic imbalance handling. Mutually exclusive with scale_pos_weight — never set both together.

In [14]:
model_isunbal = lgb.LGBMClassifier(
    random_state=42,
    n_estimators=100,
    deterministic=True,
    force_row_wise=True,
    is_unbalance=True
)

In [15]:
model_isunbal.fit(X_train, y_train, categorical_feature=categorical_cols)

[LightGBM] [Info] Number of positive: 214879, number of negative: 861401
[LightGBM] [Info] Total Bins 5648
[LightGBM] [Info] Number of data points in the train set: 1076280, number of used features: 47
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.199650 -> initscore=-1.388485
[LightGBM] [Info] Start training from score -1.388485


,random_state,42
,deterministic,True
,force_row_wise,True
,is_unbalance,True
,boosting_type,'gbdt'
,num_leaves,31
,max_depth,-1
,learning_rate,0.1
,n_estimators,100
,subsample_for_bin,200000
,objective,None


In [16]:
y_pred_proba_isunbal = model_isunbal.predict_proba(X_test)[:, 1]
y_pred_isunbal = model_isunbal.predict(X_test)

In [18]:
auc_isunbal = roc_auc_score(y_test, y_pred_proba_isunbal)
print(f"AUC: {auc_isunbal:.4f}")
print("\n")
print(confusion_matrix(y_test, y_pred_isunbal))
print("\n")
print(classification_report(y_test, y_pred_isunbal))

AUC: 0.7224


[[139306  76044]
 [ 17573  36147]]


              precision    recall  f1-score   support

           0       0.89      0.65      0.75    215350
           1       0.32      0.67      0.44     53720

    accuracy                           0.65    269070
   macro avg       0.61      0.66      0.59    269070
weighted avg       0.78      0.65      0.69    269070



## Approach 3: SMOTE

Synthetic Minority Oversampling — generates synthetic default examples by interpolating between existing ones, applied ONLY to training data (never test data, to avoid leakage).

Requires numeric input, so categorical columns are label-encoded first (separately from the category dtype used elsewhere).

In [20]:
pip install imbalanced-learn

  Using cached imbalanced_learn-0.14.2-py3-none-any.whl.metadata (8.9 kB)
  Using cached sklearn_compat-0.1.6-py3-none-any.whl.metadata (22 kB)
Using cached imbalanced_learn-0.14.2-py3-none-any.whl (236 kB)
Using cached sklearn_compat-0.1.6-py3-none-any.whl (22 kB)

   ---------------------------------------- 0/2 [sklearn-compat]
   -------------------- ------------------- 1/2 [imbalanced-learn]
   -------------------- ------------------- 1/2 [imbalanced-learn]
   -------------------- ------------------- 1/2 [imbalanced-learn]
   -------------------- ------------------- 1/2 [imbalanced-learn]
   -------------------- ------------------- 1/2 [imbalanced-learn]
   -------------------- ------------------- 1/2 [imbalanced-learn]
   -------------------- ------------------- 1/2 [imbalanced-learn]
   -------------------- ------------------- 1/2 [imbalanced-learn]
   ---------------------------------------- 2/2 [imbalanced-learn]

Note: you may need to restart the kernel to use updated packages

In [21]:
from imblearn.over_sampling import SMOTE

print("Installing/checking imblearn...")

Installing/checking imblearn...


In [22]:
from sklearn.preprocessing import LabelEncoder

In [34]:
print("Train term unique values:", X_train['term'].unique())
print("Test term unique values:", X_test['term'].unique())

train_vals = set(X_train['term'].astype(str).unique())
test_vals = set(X_test['term'].astype(str).unique())
print("In test but not train:", test_vals - train_vals)

Train term unique values: [' 36 months', ' 60 months']
Categories (2, str): [' 36 months', ' 60 months']
Test term unique values: [' 36 months', ' 60 months']
Categories (2, str): [' 36 months', ' 60 months']
In test but not train: set()


In [35]:
print(encoders['term'].classes_)

['0' '1']


In [36]:
X_train_encoded = X_train.copy()
encoders = {}

for col in categorical_cols:
    le = LabelEncoder()
    X_train_encoded[col] = le.fit_transform(X_train_encoded[col].astype(str).str.strip())
    encoders[col] = le

print(encoders['term'].classes_)

['36 months' '60 months']


In [40]:
smote = SMOTE(random_state = 42)

In [41]:
X_train_smote, y_train_smote = smote.fit_resample(X_train_encoded, y_train)

print(X_train_smote.shape)
print(y_train_smote.value_counts())

(1722802, 47)
default
0    861401
1    861401
Name: count, dtype: int64


In [42]:
model_smote = lgb.LGBMClassifier(
    random_state=42,
    n_estimators=100,
    deterministic=True,
    force_row_wise=True
)

In [43]:
model_smote.fit(X_train_smote, y_train_smote)
print("Training complete")

[LightGBM] [Info] Number of positive: 861401, number of negative: 861401
[LightGBM] [Info] Total Bins 9024
[LightGBM] [Info] Number of data points in the train set: 1722802, number of used features: 47
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
Training complete


In [44]:
X_test_encoded = X_test.copy()

for col in categorical_cols:
    X_test_encoded[col] = encoders[col].transform(X_test_encoded[col].astype(str).str.strip())

In [45]:
y_pred_proba_smote = model_smote.predict_proba(X_test_encoded)[:, 1]
y_pred_smote = model_smote.predict(X_test_encoded)

In [46]:
auc_smote = roc_auc_score(y_test, y_pred_proba_smote)

print(f"AUC: {auc_smote:.4f}")
print("\n")
print(confusion_matrix(y_test, y_pred_smote))
print("\n")
print(classification_report(y_test, y_pred_smote))

AUC: 0.7113


[[212362   2988]
 [ 49836   3884]]


              precision    recall  f1-score   support

           0       0.81      0.99      0.89    215350
           1       0.57      0.07      0.13     53720

    accuracy                           0.80    269070
   macro avg       0.69      0.53      0.51    269070
weighted avg       0.76      0.80      0.74    269070



In [47]:
joblib.dump(model_spw, '../models/scale_pos_weight_lightgbm.pkl')
print("Saved scale_pos_weight model")

Saved scale_pos_weight model


In [48]:
joblib.dump(encoders, '../models/label_encoders.pkl')

['../models/label_encoders.pkl']

## Task 4 Summary: Imbalance Handling Comparison

| Approach | AUC | Recall (default) | Precision (default) |
|---|---|---|---|
| Baseline (no handling) | 0.7166 | 0.08 | 0.58 |
| scale_pos_weight (~4.01) | 0.7224 | 0.67 | 0.32 |
| is_unbalance=True | 0.7224 | 0.67 | 0.32 |
| SMOTE | 0.7113 | 0.07 | 0.57 |

**Chosen approach: scale_pos_weight**
- Matches is_unbalance exactly, so either works — scale_pos_weight is more explicit/auditable
- Dramatically improves recall on defaults (0.08 → 0.67) at the cost of precision — an acceptable tradeoff in credit risk, since a missed default (false negative) typically costs the full loan amount, while a false positive only costs one loan's profit margin
- SMOTE underperformed, likely due to label-encoding categorical columns before interpolation, producing nonsensical synthetic categorical values. SMOTENC would be the correct fix if revisited later.
- SMOTE also took significantly longer to train (~minutes vs. near-instant for scale_pos_weight/is_unbalance) — a real scalability drawback